<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Fractal001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fftn, fftshift

def initialize_field(N, noise_level=0.1):
    # Gaussian noise + boundary damping
    field = noise_level * np.random.randn(N, N, N)
    damping = np.linspace(0, 1, N)
    damping_3d = np.outer(np.outer(damping, damping), damping).reshape(N, N, N)
    return field * damping_3d

def evolve_HES(field, dt=0.01, steps=100):
    # Placeholder for HES update rule
    for _ in range(steps):
        laplacian = np.roll(field, 1, axis=0) + np.roll(field, -1, axis=0) \
                  + np.roll(field, 1, axis=1) + np.roll(field, -1, axis=1) \
                  + np.roll(field, 1, axis=2) + np.roll(field, -1, axis=2) - 6 * field
        field += dt * laplacian
    return field

def compute_power_spectrum(field):
    spectrum = np.abs(fftshift(fftn(field)))**2
    return spectrum

def extract_dominant_wavelength(spectrum):
    # Radial average to find dominant wavelength
    N = spectrum.shape[0]
    center = N // 2
    r = np.sqrt((np.indices(spectrum.shape) - center)**2).sum(axis=0)
    r = r.astype(int)
    radial_profile = np.bincount(r.ravel(), spectrum.ravel()) / np.bincount(r.ravel())
    peak_index = np.argmax(radial_profile)
    return peak_index

def run_fractal001():
    resolutions = [25, 50, 100, 200]
    λ_dom_results = {}

    for N in resolutions:
        print(f"\nRunning simulation for N = {N}")
        field = initialize_field(N)
        field = evolve_HES(field)
        spectrum = compute_power_spectrum(field)
        λ_dom = extract_dominant_wavelength(spectrum)
        λ_dom_results[N] = λ_dom

        print(f"Dominant wavelength λ_dom at N={N}: {λ_dom}")

        # Plot and save spectrum
        plt.figure(figsize=(6, 5))
        plt.imshow(np.log1p(spectrum[N//2]), cmap='viridis')
        plt.title(f"Power Spectrum Slice at N={N}")
        plt.colorbar(label='log(Power)')
        plt.savefig(f"spectrum_N{N}.png")
        plt.close()

    print("\nSummary of λ_dom across scales:")
    for N, λ in λ_dom_results.items():
        print(f"  N = {N}: λ_dom = {λ}")

run_fractal001()



Running simulation for N = 25
Dominant wavelength λ_dom at N=25: 1

Running simulation for N = 50
Dominant wavelength λ_dom at N=50: 0

Running simulation for N = 100
Dominant wavelength λ_dom at N=100: 1

Running simulation for N = 200
Dominant wavelength λ_dom at N=200: 2

Summary of λ_dom across scales:
  N = 25: λ_dom = 1
  N = 50: λ_dom = 0
  N = 100: λ_dom = 1
  N = 200: λ_dom = 2
